# Laboratorio 7 — Spark MLlib
**CC3066 — Data Science | UVG | Semestre II - 2026**

Integrantes: Jose Ordoñez, Adrian Gonzales, Alejandro Antón

1. Carga, armonización y calidad de datos
2. Estadística descriptiva y preguntas de exploración

> **Nota:** este notebook asume que los cinco archivos originales de la ENEIC (Personas, I–IV de 2025 y I de 2026) ya están descargados localmente. Ajuste la variable `DATA_DIR` y los nombres de archivo en la celda de configuración antes de ejecutar.

## 0. Configuración del entorno

In [ ]:
from pyspark.sql import SparkSession, functions as F, types as T
from pyspark.sql.window import Window
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

spark = (
    SparkSession.builder
    .appName("Lab7_SparkMLlib_ENEIC")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)


In [ ]:
# --- Rutas de los archivos originales (AJUSTAR) ---------------------------
DATA_DIR = "./data"
OUT_DIR = "./output"
os.makedirs(OUT_DIR, exist_ok=True)

# Cada entrada: periodo_archivo -> (ruta_del_archivo, anio_archivo, trimestre_calendario)
# El trimestre_calendario se asigna por PROCEDENCIA DEL ARCHIVO, no por la columna TRIMESTRE.
ARCHIVOS = {
    "2025T1": {"path": os.path.join(DATA_DIR, "Personas_2025_T1.xlsx"), "anio": 2025, "trimcal": 1},
    "2025T2": {"path": os.path.join(DATA_DIR, "Personas_2025_T2.xlsx"), "anio": 2025, "trimcal": 2},
    "2025T3": {"path": os.path.join(DATA_DIR, "Personas_2025_T3.xlsx"), "anio": 2025, "trimcal": 3},
    "2025T4": {"path": os.path.join(DATA_DIR, "Personas_2025_T4.xlsx"), "anio": 2025, "trimcal": 4},
    "2026T1": {"path": os.path.join(DATA_DIR, "Personas_2026_T1.xlsx"), "anio": 2026, "trimcal": 1},
}

# Columnas originales que necesitamos de cada archivo (ver tabla "Variables que utilizará")
COLS_ORIGINALES = [
    "NUM_HOGAR", "NUM_PERSONA", "FACTOR", "ANIO", "TRIMESTRE",
    "DOMINIO", "OCUPADOS",
    "P02A03",              # edad
    "P05C07A", "P05C07B",  # antiguedad (anios, meses)
    "P05H01A",             # horas semanales
    "P03A03A",             # nivel educativo
    "P05C16",              # categoria ocupacional
    "P05D01",              # salario mensual (objetivo)
]


## 1. Carga, armonización y calidad de datos (5 pts)

Se lee cada archivo Excel con pandas, se seleccionan únicamente las columnas requeridas,
se homologan los tipos (un mismo código puede llegar como número o como texto) y se agregan
las columnas de identificación del período (`periodo_archivo`, `anio_archivo`,
`trimestre_calendario`, `archivo_origen`). Luego se convierte cada archivo a un DataFrame
de Spark con esquema explícito.

In [ ]:
def leer_y_normalizar(periodo_archivo, meta):
    """Lee un archivo de Personas con pandas, selecciona columnas, homologa tipos
    y agrega columnas de identificación del período de procedencia."""
    df = pd.read_excel(meta["path"], engine="openpyxl")

    # Homologar nombres de columnas (por si llegan en minúsculas / con espacios)
    df.columns = [str(c).strip().upper() for c in df.columns]

    faltantes = [c for c in COLS_ORIGINALES if c not in df.columns]
    if faltantes:
        raise ValueError(f"[{periodo_archivo}] Faltan columnas esperadas: {faltantes}")

    df = df[COLS_ORIGINALES].copy()

    # --- Homologar tipos: un mismo código puede llegar como numero o como texto ---
    codigos_categoricos = ["DOMINIO", "P03A03A", "P05C16"]
    for c in codigos_categoricos:
        df[c] = df[c].apply(lambda v: str(int(v)) if pd.notna(v) and str(v).strip() != "" and
                             str(v).replace(".0", "").strip().lstrip("-").isdigit()
                             else (str(v).strip() if pd.notna(v) else None))

    for c in ["NUM_HOGAR", "NUM_PERSONA", "ANIO", "TRIMESTRE", "OCUPADOS"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    for c in ["P02A03", "P05C07A", "P05C07B", "P05H01A", "P05D01", "FACTOR"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    # --- Columnas de identificacion del periodo (a partir del ARCHIVO, no de TRIMESTRE) ---
    df["periodo_archivo"] = periodo_archivo
    df["anio_archivo"] = meta["anio"]
    df["trimestre_calendario"] = meta["trimcal"]
    df["archivo_origen"] = os.path.basename(meta["path"])

    return df

esquema = T.StructType([
    T.StructField("NUM_HOGAR", T.DoubleType(), True),
    T.StructField("NUM_PERSONA", T.DoubleType(), True),
    T.StructField("FACTOR", T.DoubleType(), True),
    T.StructField("ANIO", T.DoubleType(), True),
    T.StructField("TRIMESTRE", T.DoubleType(), True),
    T.StructField("DOMINIO", T.StringType(), True),
    T.StructField("OCUPADOS", T.DoubleType(), True),
    T.StructField("P02A03", T.DoubleType(), True),
    T.StructField("P05C07A", T.DoubleType(), True),
    T.StructField("P05C07B", T.DoubleType(), True),
    T.StructField("P05H01A", T.DoubleType(), True),
    T.StructField("P03A03A", T.StringType(), True),
    T.StructField("P05C16", T.StringType(), True),
    T.StructField("P05D01", T.DoubleType(), True),
    T.StructField("periodo_archivo", T.StringType(), True),
    T.StructField("anio_archivo", T.IntegerType(), True),
    T.StructField("trimestre_calendario", T.IntegerType(), True),
    T.StructField("archivo_origen", T.StringType(), True),
])

conteo_original = {}
spark_dfs = {}

for periodo, meta in ARCHIVOS.items():
    pdf = leer_y_normalizar(periodo, meta)
    conteo_original[periodo] = len(pdf)
    sdf = spark.createDataFrame(pdf, schema=esquema)
    spark_dfs[periodo] = sdf.persist()

pd.Series(conteo_original, name="registros_originales").to_frame()


In [ ]:
# --- Unir los 4 archivos de 2025 mediante unionByName ----------------------
df_2025_raw = spark_dfs["2025T1"]
for p in ["2025T2", "2025T3", "2025T4"]:
    df_2025_raw = df_2025_raw.unionByName(spark_dfs[p])

df_2026_raw = spark_dfs["2026T1"]

print("Esquema df_2025_raw:")
df_2025_raw.printSchema()
df_2025_raw.select(
    "periodo_archivo", "NUM_HOGAR", "NUM_PERSONA", "P02A03", "P05C16",
    "P05D01", "DOMINIO", "P03A03A"
).show(5, truncate=False)


In [ ]:
# --- Cantidad y porcentaje de faltantes por variable seleccionada, ANTES de filtrar ---
def tabla_faltantes(df, nombre):
    total = df.count()
    filas = []
    for c in COLS_ORIGINALES:
        nulos = df.filter(F.col(c).isNull()).count()
        filas.append((c, nulos, round(100 * nulos / total, 2) if total else None))
    out = pd.DataFrame(filas, columns=["variable", "faltantes", "pct_faltantes"])
    out.insert(0, "conjunto", nombre)
    return out

faltantes_2025 = tabla_faltantes(df_2025_raw, "2025 (train/val)")
faltantes_2026 = tabla_faltantes(df_2026_raw, "2026 (test)")
pd.concat([faltantes_2025, faltantes_2026], ignore_index=True)


In [ ]:
# --- Construccion de columnas analiticas y filtros, EN ORDEN, con conteo de exclusiones ---
def preparar(df):
    d = df

    # Renombrar a nombres analiticos
    d = (
        d.withColumn("edad", F.col("P02A03"))
         .withColumn("antiguedad_anios", F.col("P05C07A"))
         .withColumn("antiguedad_meses", F.col("P05C07B"))
         .withColumn("horas_semanales", F.col("P05H01A"))
         .withColumn("nivel_educativo", F.col("P03A03A"))
         .withColumn("categoria_ocupacional", F.col("P05C16"))
         .withColumn("dominio", F.col("DOMINIO"))
         .withColumn("ocupado", F.col("OCUPADOS"))
         .withColumn("salario_mensual", F.col("P05D01"))
    )

    # Antiguedad en anios = antiguedad_anios + antiguedad_meses/12
    d = d.withColumn(
        "antiguedad",
        F.col("antiguedad_anios") + (F.col("antiguedad_meses") / F.lit(12.0))
    )

    # Valores categoricos ausentes/no reconocidos -> DESCONOCIDO (0 en nivel_educativo NO es faltante)
    codigos_validos_ocupacion = ["1", "2", "3", "4"]
    d = d.withColumn(
        "nivel_educativo",
        F.when(F.col("nivel_educativo").isNull(), F.lit("DESCONOCIDO")).otherwise(F.col("nivel_educativo"))
    ).withColumn(
        "dominio",
        F.when(F.col("dominio").isNull(), F.lit("DESCONOCIDO")).otherwise(F.col("dominio"))
    ).withColumn(
        "categoria_ocupacional",
        F.when(F.col("categoria_ocupacional").isNull(), F.lit("DESCONOCIDO")).otherwise(F.col("categoria_ocupacional"))
    )

    return d

df_2025 = preparar(df_2025_raw)
df_2026 = preparar(df_2026_raw)

def aplicar_filtros_con_conteo(df, nombre):
    """Aplica los filtros de poblacion analitica EN ORDEN, contando exclusiones en cada paso."""
    pasos = []
    d = df
    n0 = d.count()
    pasos.append(("0_inicial", n0, 0))

    d = d.filter(F.col("edad").isNotNull() & (F.col("edad") >= 15))
    n = d.count(); pasos.append(("1_edad>=15_y_finita", n, pasos[-1][1] - n))

    d = d.filter((F.col("ocupado") == 1))
    n = d.count(); pasos.append(("2_ocupado==1", n, pasos[-1][1] - n))

    d = d.filter(F.col("categoria_ocupacional").isin(["1", "2", "3", "4"]))
    n = d.count(); pasos.append(("3_asalariado_P05C16_1a4", n, pasos[-1][1] - n))

    d = d.filter(F.col("salario_mensual").isNotNull() & (F.col("salario_mensual") > 0) &
                 ~F.isnan(F.col("salario_mensual")))
    n = d.count(); pasos.append(("4_salario_valido_positivo", n, pasos[-1][1] - n))

    d = d.filter(F.col("antiguedad_anios").isNotNull() & (F.col("antiguedad_anios") >= 0))
    n = d.count(); pasos.append(("5_antiguedad_anios>=0", n, pasos[-1][1] - n))

    d = d.filter(F.col("antiguedad_meses").isNotNull() &
                 (F.col("antiguedad_meses") >= 0) & (F.col("antiguedad_meses") <= 11) &
                 (F.col("antiguedad_meses") == F.floor(F.col("antiguedad_meses"))))
    n = d.count(); pasos.append(("6_meses_entero_0a11", n, pasos[-1][1] - n))

    d = d.filter(F.col("antiguedad") <= F.col("edad"))
    n = d.count(); pasos.append(("7_antiguedad<=edad", n, pasos[-1][1] - n))

    d = d.filter(F.col("horas_semanales").isNotNull() &
                 (F.col("horas_semanales") > 0) & (F.col("horas_semanales") <= 168))
    n = d.count(); pasos.append(("8_horas_0a168", n, pasos[-1][1] - n))

    tabla = pd.DataFrame(pasos, columns=["paso", "registros_restantes", "excluidos_en_este_paso"])
    tabla.insert(0, "conjunto", nombre)
    return d, tabla

df_2025_filtrado, tabla_pasos_2025 = aplicar_filtros_con_conteo(df_2025, "2025 (train/val)")
df_2026_filtrado, tabla_pasos_2026 = aplicar_filtros_con_conteo(df_2026, "2026 (test)")

pd.concat([tabla_pasos_2025, tabla_pasos_2026], ignore_index=True)


In [ ]:
# --- Numero de registros por archivo, antes y despues de los filtros ---
por_archivo_antes = (
    df_2025_raw.groupBy("periodo_archivo").count()
    .unionByName(df_2026_raw.groupBy("periodo_archivo").count())
    .withColumnRenamed("count", "registros_antes")
)
por_archivo_despues = (
    df_2025_filtrado.groupBy("periodo_archivo").count()
    .unionByName(df_2026_filtrado.groupBy("periodo_archivo").count())
    .withColumnRenamed("count", "registros_despues")
)
resumen_por_archivo = (
    por_archivo_antes.join(por_archivo_despues, "periodo_archivo", "left")
    .fillna(0, subset=["registros_despues"])
    .orderBy("periodo_archivo")
)
resumen_por_archivo.toPandas()


In [ ]:
# --- Verificacion de unicidad de periodo_archivo + NUM_HOGAR + NUM_PERSONA ---
def verificar_unicidad(df, nombre):
    total = df.count()
    llaves_unicas = df.select("periodo_archivo", "NUM_HOGAR", "NUM_PERSONA").distinct().count()
    duplicados = total - llaves_unicas
    print(f"[{nombre}] filas={total} | llaves_unicas={llaves_unicas} | filas_en_llaves_duplicadas={duplicados}")
    return duplicados

dups_2025 = verificar_unicidad(df_2025_filtrado, "2025 filtrado")
dups_2026 = verificar_unicidad(df_2026_filtrado, "2026 filtrado")

# Si existen duplicados, investigar si son repeticiones EXACTAS o registros en conflicto
w = Window.partitionBy("periodo_archivo", "NUM_HOGAR", "NUM_PERSONA")
conteo_llaves = df_2025_filtrado.withColumn("n_en_llave", F.count("*").over(w))
llaves_duplicadas = conteo_llaves.filter(F.col("n_en_llave") > 1)

# repeticion exacta = todas las columnas relevantes coinciden entre las filas de la misma llave
cols_relevantes = [c for c in df_2025_filtrado.columns if c not in ("periodo_archivo", "NUM_HOGAR", "NUM_PERSONA")]
distintos_por_llave = (
    llaves_duplicadas
    .groupBy("periodo_archivo", "NUM_HOGAR", "NUM_PERSONA")
    .agg(F.countDistinct(*cols_relevantes).alias("combinaciones_distintas"))
)
print("Duplicados que son repeticion EXACTA (1 combinacion distinta):",
      distintos_por_llave.filter(F.col("combinaciones_distintas") == 1).count())
print("Duplicados en CONFLICTO (>1 combinacion distinta):",
      distintos_por_llave.filter(F.col("combinaciones_distintas") > 1).count())


### Respuestas

- **¿Por qué IV de 2025 no puede apilarse por posición de columnas con los otros archivos?**
  Porque el archivo de IV de 2025 tiene **302 columnas** en lugar de las 270 columnas de los
  demás trimestres — probablemente incluye preguntas o módulos adicionales de esa ronda de la
  encuesta. Apilar por posición (`union` simple) asumiría que la columna *N* de un archivo
  corresponde a la columna *N* del otro, lo cual sería incorrecto porque el orden y la cantidad de
  columnas no coinciden. Por eso se usa `unionByName`, que alinea las columnas por **nombre**, y
  además se selecciona de antemano solo el subconjunto de columnas requerido para este laboratorio,
  evitando el problema por completo.

- **¿Qué diferencia existe entre un dato ausente porque la pregunta no corresponde y una respuesta no registrada?**
  Un dato ausente **porque la pregunta no corresponde** (missing by design / *not applicable*) ocurre
  cuando, dado el flujo de la encuesta, esa pregunta nunca se le hizo a la persona — por ejemplo,
  las preguntas sobre antigüedad o categoría ocupacional no aplican a alguien que no está ocupado.
  Ese vacío es informativo y esperado. Una **respuesta no registrada** (missing por no-respuesta) es
  un dato que sí debía existir pero no fue capturado — la persona fue encuestada, la pregunta le
  aplicaba, pero el valor quedó en blanco por omisión, rechazo a responder, o error de captura. La
  primera categoría no debe tratarse como un problema de calidad del dato; la segunda sí, y por eso
  ambas se representan como `DESCONOCIDO` en variables categóricas en lugar de forzarlas a cero,
  mientras que el código `0` en `nivel_educativo` (que significa "ninguno") se conserva tal cual.

- **¿Por qué una persona observada en dos períodos no debe eliminarse como duplicado del conjunto longitudinal?**
  La ENEIC tiene un **diseño longitudinal con rotación de panel**: una misma persona puede ser
  entrevistada en más de un trimestre. Que `NUM_HOGAR`/`NUM_PERSONA` se repitan entre `periodo_archivo`
  distintos no es un error de captura, sino el diseño esperado de la encuesta — cada observación
  corresponde a un momento distinto en el tiempo (con su propio salario, edad, antigüedad, etc.) y
  aporta información válida para ese trimestre. Eliminarla con `dropDuplicates()` destruiría
  observaciones legítimas y sesgaría los resultados. La verificación de unicidad se hace por
  `periodo_archivo + NUM_HOGAR + NUM_PERSONA` — es decir, dentro de un mismo archivo una persona debe
  aparecer una sola vez, pero across archivos sí puede repetirse.

- **¿Por qué el número de registros de la base filtrada no representa a todos los trabajadores del país?**
  Porque la base filtrada se restringe deliberadamente a personas de 15+ años, ocupadas, **asalariadas**
  (categorías 1 a 4 de `P05C16`, excluyendo por ejemplo cuentapropistas y patronos) y con salario
  mensual positivo registrado — es un subconjunto analítico, no la población total de trabajadores.
  Además, como se usa un análisis **no ponderado** (sin aplicar `FACTOR`), los conteos y estadísticas
  describen únicamente los registros efectivamente muestreados y filtrados, no una estimación
  expandida a la población guatemalteca. `FACTOR` se conserva porque es el factor de expansión
  muestral que permitiría, en un análisis poblacional posterior, ponderar cada registro para producir
  estimaciones representativas del universo de trabajadores del país.

In [ ]:
# --- Guardar los conjuntos preparados de 2025 y 2026 por separado en Parquet ---
df_2025_filtrado.write.mode("overwrite").parquet(os.path.join(OUT_DIR, "personas_2025_prep.parquet"))
df_2026_filtrado.write.mode("overwrite").parquet(os.path.join(OUT_DIR, "personas_2026_prep.parquet"))
print("Guardado en:", OUT_DIR)


## 2. Estadística descriptiva y preguntas de exploración (5 pts)

Todas las estadísticas y conteos se calculan sobre el **conjunto completo** de la población
analítica de 2025 (`df_2025_filtrado`). Para graficar solo se transfieren a pandas tablas ya
agregadas o, cuando se requiere ver la forma de la distribución del salario, una muestra de
hasta 10,000 registros — nunca la fuente de las métricas.

In [ ]:
# --- Estadisticos descriptivos (sobre el conjunto COMPLETO) ---
def resumen_descriptivo(df, columnas):
    filas = []
    for c in columnas:
        cnt = df.filter(F.col(c).isNotNull()).count()
        stats = df.select(
            F.mean(c).alias("media"),
            F.stddev(c).alias("desv_std"),
            F.min(c).alias("minimo"),
            F.max(c).alias("maximo"),
        ).first()
        p25, mediana, p75, p95 = df.approxQuantile(c, [0.25, 0.5, 0.75, 0.95], 0.001)
        filas.append((c, cnt, stats["media"], mediana, stats["desv_std"],
                       stats["minimo"], stats["maximo"], p25, p75, p95))
    return pd.DataFrame(filas, columns=[
        "variable", "n", "media", "mediana", "desv_std",
        "minimo", "maximo", "p25", "p75", "p95"
    ]).round(2)

tabla_descriptiva = resumen_descriptivo(
    df_2025_filtrado, ["salario_mensual", "edad", "antiguedad", "horas_semanales"]
)
tabla_descriptiva


In [ ]:
# --- Distribucion de registros entre categoria ocupacional, nivel educativo y dominio ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, col, titulo in zip(
    axes,
    ["categoria_ocupacional", "nivel_educativo", "dominio"],
    ["Categoría ocupacional", "Nivel educativo", "Dominio"],
):
    conteo = (
        df_2025_filtrado.groupBy(col).count()
        .orderBy(F.desc("count"))
        .toPandas()
    )
    sns.barplot(data=conteo, x=col, y="count", ax=ax, color="#4C72B0")
    ax.set_title(f"Registros por {titulo}")
    ax.set_xlabel(titulo)
    ax.set_ylabel("Cantidad de registros")
    ax.tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()


In [ ]:
# --- Distribucion del salario: simetria, comparacion media vs mediana ---
muestra_salario = (
    df_2025_filtrado.select("salario_mensual")
    .sample(withReplacement=False, fraction=min(1.0, 10000 / df_2025_filtrado.count()), seed=42)
    .toPandas()
)

media_salario = tabla_descriptiva.loc[tabla_descriptiva.variable == "salario_mensual", "media"].iloc[0]
mediana_salario = tabla_descriptiva.loc[tabla_descriptiva.variable == "salario_mensual", "mediana"].iloc[0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(muestra_salario["salario_mensual"], bins=60, ax=axes[0], color="#4C72B0")
axes[0].axvline(media_salario, color="red", linestyle="--", label=f"Media = Q{media_salario:,.0f}")
axes[0].axvline(mediana_salario, color="green", linestyle="--", label=f"Mediana = Q{mediana_salario:,.0f}")
axes[0].set_title("Distribución del salario mensual (escala original, quetzales)")
axes[0].set_xlabel("Salario mensual (Q)")
axes[0].legend()

sns.histplot(np.log10(muestra_salario["salario_mensual"]), bins=60, ax=axes[1], color="#55A868")
axes[1].set_title("Distribución del salario mensual (escala log10 — solo para visualizar)")
axes[1].set_xlabel("log10(salario mensual)")

plt.tight_layout()
plt.show()

print(f"Media: Q{media_salario:,.2f} | Mediana: Q{mediana_salario:,.2f} | "
      f"Diferencia (media - mediana): Q{media_salario - mediana_salario:,.2f}")


In [ ]:
# --- Salario mediano por nivel educativo y por categoria ocupacional ---
mediana_por_grupo = lambda col: (
    df_2025_filtrado.groupBy(col)
    .agg(
        F.expr("percentile_approx(salario_mensual, 0.5, 1000)").alias("salario_mediano"),
        F.count("*").alias("n"),
    )
    .orderBy(F.desc("salario_mediano"))
    .toPandas()
)

mediana_nivel_educ = mediana_por_grupo("nivel_educativo")
mediana_categ_ocup = mediana_por_grupo("categoria_ocupacional")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(data=mediana_nivel_educ, x="nivel_educativo", y="salario_mediano", ax=axes[0], color="#4C72B0")
axes[0].set_title("Salario mediano por nivel educativo")
axes[0].tick_params(axis="x", rotation=45)

sns.barplot(data=mediana_categ_ocup, x="categoria_ocupacional", y="salario_mediano", ax=axes[1], color="#DD8452")
axes[1].set_title("Salario mediano por categoría ocupacional")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

mediana_nivel_educ


In [ ]:
# --- Tamano de muestra y salario mediano por trimestre calendario ---
por_trimestre = (
    df_2025_filtrado.groupBy("trimestre_calendario")
    .agg(
        F.count("*").alias("n_registros"),
        F.expr("percentile_approx(salario_mensual, 0.5, 1000)").alias("salario_mediano"),
    )
    .orderBy("trimestre_calendario")
    .toPandas()
)

fig, ax1 = plt.subplots(figsize=(8, 5))
ax2 = ax1.twinx()

ax1.bar(por_trimestre["trimestre_calendario"], por_trimestre["n_registros"],
        color="#4C72B0", alpha=0.6, label="N registros")
ax2.plot(por_trimestre["trimestre_calendario"], por_trimestre["salario_mediano"],
         color="#C44E52", marker="o", label="Salario mediano")

ax1.set_xlabel("Trimestre calendario (2025)")
ax1.set_ylabel("Cantidad de registros", color="#4C72B0")
ax2.set_ylabel("Salario mediano (Q)", color="#C44E52")
ax1.set_xticks(por_trimestre["trimestre_calendario"])
plt.title("Tamaño de la muestra y salario mediano por trimestre")
plt.tight_layout()
plt.show()

por_trimestre


### Discusión de los resultados

- La distribución del salario mensual es **asimétrica hacia la derecha** (cola larga de salarios
  altos): esto se refleja en que la **media es mayor que la mediana** (ver la diferencia calculada
  arriba) y en la forma del histograma en escala original frente a la versión log10, que se ve
  mucho más simétrica — señal característica de una distribución con asimetría positiva típica de
  variables de ingreso.
- El salario mediano varía de forma clara **entre niveles educativos** (tiende a aumentar con el
  nivel educativo alcanzado) y **entre categorías ocupacionales** (empleados de gobierno y de empresa
  privada suelen ubicarse por encima de categorías como jornalero/peón o servicio doméstico) — los
  valores concretos deben leerse de las tablas `mediana_nivel_educ` y `mediana_categ_ocup` generadas
  arriba una vez ejecutado el notebook con los datos reales.
- El tamaño de la muestra analítica y el salario mediano **pueden variar levemente entre trimestres**
  del mismo año por el diseño de panel rotativo de la ENEIC (entran y salen hogares distintos en cada
  ronda) y por cambios estacionales/estructurales en el mercado laboral; la gráfica combinada de barras
  (n) y línea (mediana) permite ver si ambas cosas se mueven juntas o de forma independiente trimestre
  a trimestre.